In [1]:
pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=9d3703cc89f8d086d0b13b5dc946cfa35759f6d1e1deb821fcc6e398b133183b
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [2]:
import requests
import pandas as pd
import time
from langdetect import detect, DetectorFactory

In [3]:
DetectorFactory.seed = 0

In [4]:
GITHUB_TOKEN = "x"

headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

In [5]:
def paginate(url, params, max_results=500):
    results = []
    page = 1

    while len(results) < max_results:
        params.update({"page": page, "per_page": 100})
        time.sleep(1.2)  #delay so that github doesnt block us

        response = requests.get(url, headers=headers, params=params)

        # gitHub secondary rate limit
        if response.status_code == 403:
            print("\n waiting 60 seconds for Github secondary limit \n")
            time.sleep(60)
            continue

        # if the query > 1000 skip
        if response.status_code == 422:
            print("skipping:", params["q"])
            return []

        if response.status_code != 200:
            print("Error:", response.json())
            return []

        data = response.json()
        items = data.get("items", [])
        if not items:
            break

        results.extend(items)
        page += 1

    return results[:max_results]


In [6]:
def search_repositories(query, year=None):
    q = query
    if year:
        q += f" created:>={year}-01-01"

    return paginate("https://api.github.com/search/repositories", {"q": q})

In [7]:
def search_issues(query, year=None):
    q = query
    if year:
        q += f" created:>={year}-01-01"

    return paginate("https://api.github.com/search/issues", {"q": q})

In [8]:
def search_code(query):
    return paginate("https://api.github.com/search/code", {"q": query})

In [9]:
def is_english(text):
    """Returns True if the text is English, False otherwise."""
    if not text or not text.strip():
        return True  # empty = assume English (avoid false negatives)
    try:
        lang = detect(text)
        return lang == "en"
    except:
        return True

In [10]:
def run_multiple_queries(queries, year=None):
    all_rows = []

    for q in queries:
        print(f"query: {q}")
        time.sleep(3)

        repo_results = search_repositories(q, year)
        issues_results = search_issues(q, year)
        code_results = search_code(q)

        for repo in repo_results:
            desc = repo.get("description", "") or ""
            if not is_english(desc):
                continue  # skip non-english repos

            all_rows.append({
                "type": "repository",
                "query": q,
                "name": repo.get("full_name"),
                "url": repo.get("html_url"),
                "description": desc,
                "stars": repo.get("stargazers_count", 0),
                "language": repo.get("language", ""),
                "date": repo.get("updated_at")
            })

        # Issues with English repos
        for item in issues_results:
            desc = item.get("body", "") or ""
            if not is_english(desc):
                continue  # skip

            all_rows.append({
                "type": "issue",
                "query": q,
                "name": item.get("title"),
                "url": item.get("html_url"),
                "description": desc,
                "stars": None,
                "language": None,
                "date": item.get("updated_at")
            })

        # code search for english repos
        for item in code_results:
            repo = item.get("repository", {})
            desc = "Code search result"

            # English by definition
            if not is_english(desc):
                continue

            all_rows.append({
                "type": "code",
                "query": q,
                "name": f"{repo.get('full_name', '')} / {item.get('name', '')}",
                "url": item.get("html_url"),
                "description": desc,
                "stars": repo.get("stargazers_count", 0),
                "language": repo.get("language", ""),
                "date": repo.get("updated_at")
            })

    return all_rows

In [11]:
# Keyword bucket

oss_terms = [
    "open source",
    "open-source",
    "OSS",
    "FOSS",
    "open source software"
]

ai_terms = [
    "artificial intelligence",
    "AI",
    "machine learning",
    "ML",
    "deep learning",
    "neural network",
    "large language model",
    "LLM",
    "generative AI"
]

security_terms = [
    "security",
    "cybersecurity",
    "cyber security",
    "adversarial",
    "adversarial attack",
    "robustness",
    "secure",
    "ai safety",
    "secure ai",
    "model evaluation",
    "risk",
    "threat"
]


In [12]:
# build combined queries (mix of keywords)
queries = []

for oss in oss_terms:
    for ai in ai_terms:
        for sec in security_terms:
            q = f"\"{oss}\" \"{ai}\" \"{sec}\" stars:>50 in:name,description,readme"
            queries.append(q)


queries = list(dict.fromkeys(queries))

print(f"Number of queries: {len(queries)}")
for q in queries[:10]:
    print(q)


Number of queries: 540
"open source" "artificial intelligence" "security" stars:>50 in:name,description,readme
"open source" "artificial intelligence" "cybersecurity" stars:>50 in:name,description,readme
"open source" "artificial intelligence" "cyber security" stars:>50 in:name,description,readme
"open source" "artificial intelligence" "adversarial" stars:>50 in:name,description,readme
"open source" "artificial intelligence" "adversarial attack" stars:>50 in:name,description,readme
"open source" "artificial intelligence" "robustness" stars:>50 in:name,description,readme
"open source" "artificial intelligence" "secure" stars:>50 in:name,description,readme
"open source" "artificial intelligence" "ai safety" stars:>50 in:name,description,readme
"open source" "artificial intelligence" "secure ai" stars:>50 in:name,description,readme
"open source" "artificial intelligence" "model evaluation" stars:>50 in:name,description,readme


In [13]:
results = run_multiple_queries(queries, year=2020)

df = pd.DataFrame(results)
df.to_csv("github_osint_results.csv", index=False)

print(f"\n {len(df)} results saved in github_osint_results.csv")


query: "open source" "artificial intelligence" "security" stars:>50 in:name,description,readme
skipping: "open source" "artificial intelligence" "security" stars:>50 in:name,description,readme
query: "open source" "artificial intelligence" "cybersecurity" stars:>50 in:name,description,readme
skipping: "open source" "artificial intelligence" "cybersecurity" stars:>50 in:name,description,readme
query: "open source" "artificial intelligence" "cyber security" stars:>50 in:name,description,readme
skipping: "open source" "artificial intelligence" "cyber security" stars:>50 in:name,description,readme
query: "open source" "artificial intelligence" "adversarial" stars:>50 in:name,description,readme
skipping: "open source" "artificial intelligence" "adversarial" stars:>50 in:name,description,readme
query: "open source" "artificial intelligence" "adversarial attack" stars:>50 in:name,description,readme
skipping: "open source" "artificial intelligence" "adversarial attack" stars:>50 in:name,descri

In [15]:
from google.colab import files
files.download("github_osint_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>